# CAFA 6 Model testing

## Purpose

This Sheet is used to train models on the varuious datasets that I generated from the available data. 

Data is located in "./data/exports"

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import obonet
import networkx
from Bio import SeqIO
import tensorflow as tf # Will return a lot of warnings, but loads just fine
import tensorflow_hub as hub
import tf_keras
from tf_keras.models import Sequential
from tf_keras.layers import Dense
from tf_keras.layers import Dropout
from tf_keras.layers import LSTM
from tf_keras.layers import RNN
from tf_keras import utils
from tf_keras.callbacks import ModelCheckpoint
from progressbar import progressbar
print(f"Tensor flow Version: {tf.__version__}")
print(f"Tensor flow hub version: {hub.__version__}")
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

# Simple test
x = tf.constant([[1, 2], [3, 4]])
print(f"Test tensor:\n{x}")

2025-12-22 12:10:33.680484: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-22 12:10:33.701761: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-22 12:10:33.707825: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-22 12:10:33.721676: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-22 12:10:34.903973: W tensorflow/compiler/tf2

Tensor flow Version: 2.17.0
Tensor flow hub version: 0.16.1
TensorFlow version: 2.17.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Test tensor:
[[1 2]
 [3 4]]


I0000 00:00:1766401836.629818   12597 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1766401836.675313   12597 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1766401836.675563   12597 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1766401836.677808   12597 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

In [2]:
# unique Proteins/go terms
train_terms = pd.read_csv("./data/cafa-6-protein-function-prediction/Train/train_terms.tsv", delimiter="\t")
UNIQUE_GO_TERMS = np.unique(train_terms["term"])
UNIQUE_PROTEINS = np.unique(train_terms["EntryID"])

In [3]:
# Read X, y are part of the .tsv. How ether, I can't really save a working string there and have it read, thus an additional dictionary..
import pickle
with open("./data/exports/proteins.pkl", "rb") as fp:
    protein_dict = pickle.load(fp)
max_protein_len = np.max(np.array(protein_dict.values()))

In [5]:
# Create the tf.lookup.StaticHashTable that we are using instead of the dict, which we imported earlier.
def create_tf_static_hash_table(input_dict):
    """
    This turns the input dictionaty into an tf.lookup.StaticHashTable with tf.Ragged tenstors as values 
    in order to make is useable by the preprocessing function.
    Params:
    **input_dict:** The dictionary, which shall be converted
    """
    # Create a TensorFlow lookup table from your dictionary
    keys = list(input_dict.keys())

    # Serialize each value as a ragged tensor with int8
    serialized_values = []
    for item in input_dict.values():
        ragged = tf.constant([item], dtype=tf.int8)
        serialized = tf.io.serialize_tensor(ragged)
        serialized_values.append(serialized.numpy())

    table = tf.lookup.StaticHashTable(
        tf.lookup.KeyValueTensorInitializer(
            keys=keys,
            values=serialized_values,
        ),
        default_value=b""
    )
    return table

protein_lookup_table = create_tf_static_hash_table(protein_dict)

In [ ]:
def create_tf_static_len_hash_table(input_dict):
    """
    This turns the input dictionary into a tf.lookup.StaticHashTable
    Params:
    **input_dict:** The dictionary, which shall be converted
    """
    keys = list(input_dict.keys())
    values = list(input_dict.values())
    
    # Find the maximum length
    max_len = np.max(np.array(values))
    
    # Pad all sequences to the same length
    padded_values = [v + [0] * (max_len - len(v)) for v in values]
    
    table = tf.lookup.StaticHashTable(
        tf.lookup.KeyValueTensorInitializer(
            keys=keys,
            values=padded_values,
        ),
        default_value=tf.zeros(max_len, dtype=tf.float32)
    )
    return table

## Provide function to read y, line by line

Code based off of https://stackoverflow.com/questions/49900798/how-to-limit-ram-usage-while-batch-training-in-tensorflow
As in using the tf.data.TextLineDataset, to load the data on demand.

### ChatGPT output:

Sure! I'll walk you through some TensorFlow code that loads data from a `.tsv` (Tab-Separated Values) file on-demand for model training. This is a common pattern when dealing with large datasets that don't fit entirely into memory at once. We will use `tf.data` API to create an efficient data pipeline that loads data on demand.

### Steps:

1. **Loading the Data**: We'll load the data from a `.tsv` file using TensorFlow’s `tf.data.experimental.CsvDataset` (which works for `.csv` but can be used for `.tsv` by specifying the appropriate delimiter).
2. **Preprocessing**: We'll process the data to extract features (input data) and labels (output data).
3. **Model Training**: We'll use this dataset for training a simple neural network model.

### Example Code:

```python
import tensorflow as tf

# Define a function to load and parse the TSV data
def load_data(file_path, batch_size=32, shuffle_buffer=10000):
    # This assumes your file has columns like: feature1, feature2, ..., label
    # For a tab-separated file, we specify the delimiter as '\t'
    dataset = tf.data.experimental.CsvDataset(
        file_path,
        [tf.float32, tf.float32, tf.float32],  # Modify based on the number and types of columns in your file
        header=True,  # Skip the header row
        field_delim='\t'  # Tab-separated values
    )
    
    # This function will separate the features (X) and the labels (y)
    def preprocess_data(features, labels):
        # You can apply any additional preprocessing steps here if needed (e.g., normalization)
        return features, labels
    
    # Preprocess and shuffle the data
    dataset = dataset.map(preprocess_data)
    dataset = dataset.shuffle(buffer_size=shuffle_buffer)
    dataset = dataset.batch(batch_size)
    
    return dataset

# Define a simple model
def create_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu', input_shape=(2,)),  # Modify input shape based on your features
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')  # For binary classification
    ])
    
    model.compile(optimizer='adam',
                  loss='binary_crossentropy',  # For binary classification
                  metrics=['accuracy'])
    
    return model

# Path to your .tsv file
file_path = 'your_data.tsv'

# Load data
dataset = load_data(file_path)

# Create and train the model
model = create_model()
model.fit(dataset, epochs=10)

```


In [6]:
def preprocess_data(*args):
    """
    This Function will perform additional preprocessing of the data.
    Parameters:
    **args** One line of the dataset, with the entryID as the first Item
    """
    entry_id = args[0]
    labels = tf.stack(args[1:])
    
    # Look up the serialized feature
    serialized_feature = protein_lookup_table.lookup(entry_id)
    
    # Deserialize back to Tensor with int8
    feature = tf.io.parse_tensor(serialized_feature, out_type=tf.int8)
    
    # Remove the outer dimension we added during serialization
    feature = feature[0]
    
    # If we need to convert to float32 for embedding layer or other operations later:
    # feature = tf.cast(feature, tf.float32)
    
    return feature, labels

In [ ]:
def load_data(file:str="./data/exports/reduced_tensors.tsv", batch_size:int=32, shuffle_buffer=10000):
    """
    This function will make and preprocess a label dataset from an external csv file in order to limit RAM useage.
    This supposed to be provides to the tensor flow setup, which trains the model, don't use individually
    Params:
    **file** Path to the file to open
    **batch_size** The batch size to use
    **shuffle_buffer** The size of the buffer for shuffling
    """
    dataset = tf.data.experimental.CsvDataset("./data/exports/reduced_tensors.tsv", 
                                                   [tf.string]+[tf.float32]*len(UNIQUE_GO_TERMS), 
                                                   field_delim="\t", 
                                                   header=True)


    # Now, Preprocessing, shuffling and Batching, allthjough it is already batched?
    dataset = dataset.map(preprocess_data).shuffle(buffer_size=shuffle_buffer, seed=42).padded_batch(
        batch_size=batch_size,
        padded_shapes=([None], [max_protein_len]),
        padding_values=(tf.constant(0, dtype=tf.int8), tf.constant(0.0, dtype=tf.float32))  # Explicitly set padding values
    )
    return dataset

In [6]:
test_dataset = tf.data.experimental.CsvDataset("./data/exports/reduced_ids.tsv", 
                                                   [tf.string]+[tf.float32]*len(UNIQUE_GO_TERMS), 
                                                   field_delim="\t", 
                                                   header=True)


In [43]:
for line in test_dataset.take(10):
    print(line)

(<tf.Tensor: shape=(), dtype=string, numpy=b'A0A0C9S461'>, <tf.Tensor: shape=(), dtype=float32, numpy=0.0>, <tf.Tensor: shape=(), dtype=float32, numpy=0.0>, <tf.Tensor: shape=(), dtype=float32, numpy=0.0>, <tf.Tensor: shape=(), dtype=float32, numpy=0.0>, <tf.Tensor: shape=(), dtype=float32, numpy=0.0>, <tf.Tensor: shape=(), dtype=float32, numpy=0.0>, <tf.Tensor: shape=(), dtype=float32, numpy=0.0>, <tf.Tensor: shape=(), dtype=float32, numpy=0.0>, <tf.Tensor: shape=(), dtype=float32, numpy=0.0>, <tf.Tensor: shape=(), dtype=float32, numpy=0.0>, <tf.Tensor: shape=(), dtype=float32, numpy=0.0>, <tf.Tensor: shape=(), dtype=float32, numpy=0.0>, <tf.Tensor: shape=(), dtype=float32, numpy=0.0>, <tf.Tensor: shape=(), dtype=float32, numpy=0.0>, <tf.Tensor: shape=(), dtype=float32, numpy=0.0>, <tf.Tensor: shape=(), dtype=float32, numpy=0.0>, <tf.Tensor: shape=(), dtype=float32, numpy=0.0>, <tf.Tensor: shape=(), dtype=float32, numpy=0.0>, <tf.Tensor: shape=(), dtype=float32, numpy=0.0>, <tf.Tensor

In [19]:
test_dataset.take(10).map(preprocess_data)

<_MapDataset element_spec=(TensorSpec(shape=<unknown>, dtype=tf.int8, name=None), TensorSpec(shape=(26125,), dtype=tf.float32, name=None))>

In [40]:
# test_dataset.element_spec
# OrderedDict([('GO:0000001', TensorSpec(shape=(32,), dtype=tf.float32, name=None)),(...)])
for line in test_dataset.take(10):
    #print([item.numpy() for item in line])
    print(np.argmax(np.array(line[1:])))

7897
13316
1927
9466
1550
8377
2831
7735
2547
824


## Model building

Now, build the model:
- We are using sequential data of variable lengths, we need RNN, in this case LSTM nets (nets with hidden internal state, to have the sorage)
- Some hidden layers
- Finally, a many individual classification output - each GO-factor needs to be predicted, individually.

In [ ]:
# Define a simple model
def create_model():# Setting up a sequential net, with the Protein input
    vocab_size = 25
    model = Sequential([
        tf_keras.layers.InputLayer(input_shape=(None,), ragged=True, dtype=tf.int8),
        tf_keras.layers.Embedding(vocab_size, 128, mask_zero=True),  # Converts to (batch, length, 128)
        # tf_keras.layers.Lambda(lambda x: x.to_tensor()),  # Convert ragged to dense
        LSTM(700, return_sequences=True),
        Dropout(0.2),
        LSTM(700, return_sequences=True),
        Dropout(0.2),
        LSTM(700),
        Dropout(0.2),
        Dense(len(UNIQUE_GO_TERMS), activation='sigmoid')
    ])
    
    model.compile(optimizer='adam',
                  loss='binary_crossentropy',  # For multi-label classification
                  metrics=['accuracy'])
    
    return model

### Create Callbacks for the model training

- Tensorboard Callback to check the model training
- Early stopping callback to stop the model on time, and prevent overfitting

In [2]:
# Load the tensorboard extention
%load_ext tensorboard

In [ ]:
import datetime
import os

# Create a function to build a TensorBoard callback
def create_tensorboard_callback():
  # Create a log directory for storing TensorBoard logs
  logdir = os.path.join("./logs",
                        # Make it so the logs get tracked whenever we run an experiment
                        datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
  return tf_keras.callbacks.TensorBoard(logdir)

In [ ]:
# Create eraly stopping (once out model stops improving, stop training)
early_stopping = tf_keras.callbacks.EarlyStopping(monitor="val_accuracy",
                                                  patience=3) # Stops after 3 rounds of no improvements

In [ ]:

# Path to your .tsv file
file_path = './data/exports/reduced_ids.tsv'

# Load data
dataset = load_data(file_path)

# Create new TensorBoard session everytime we train a model
tensorboard = create_tensorboard_callback()

# Create and train the model
model = create_model()
model.fit(X=dataset.feature, 
          y=dataset.labels,
          
          epochs=10)

NameError: name 'load_data' is not defined

In [19]:
X.shape, X.dtypes

NameError: name 'y' is not defined